# A reference resolves by code

A consumer sends a currency reference with an outdated label and symbol.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
tag = PriceTagCoreEntity.model_validate(
    {
        "id": "tag-1",
        "amount": 9.99,
        "currency": {"code": "eur", "display_name": "Old label", "properties": {"symbol": "?"}},
    }
)
print("Shared value:", tag.currency is CurrencyEnumEntity.EUR)
print(tag.currency.display_name, tag.currency.symbol)
assert tag.currency is CurrencyEnumEntity.EUR

Shared value: True
Euro €


## What the result means

The incoming code selects EUR from the catalogue. The old label and symbol are ignored: they do not overwrite the declaration and are not checked for equality. A reference containing only `{"code": "eur"}` selects the same object.